# Random forest baseline: AD vs HC on ds004504

A classical, non-graph baseline for the MFE-FCGCN and MFE-FCGAT notebooks
(`GCN_ds004504.ipynb`, `GAT_ds004504.ipynb`). It answers one question: **how
much of the AD-vs-HC signal does a standard classifier get from band power
alone, scored under exactly the same protocol?**

```
per 20 s window (75 % overlap):

  mean log10 band power   5 bands x 19 electrodes  ->  95 features
    |  RandomForestClassifier   500 trees, scikit-learn defaults
    v
  P(AD) per window  --mean over the subject's windows-->  P(AD) per subject
```

| | MFE-FCGAT (`GAT_ds004504.ipynb`) | Random forest (this notebook) |
|---|---|---|
| input per window | full PSD (227 bins x 19 electrodes) + Pearson and MI graphs | mean log band power (5 x 19 = 95 numbers) |
| model | CNN -> 2 graph-attention layers per band -> MLP | 500 decision trees, averaged votes |
| fitted on | train subjects; validation subjects pick the epoch | train + validation subjects (no epoch to pick) |
| folds, scoring, pooled metric | identical code | identical code |

**Protocol.** Same as the graph notebooks: 5-fold cross-validation grouped by
subject, with the same fold seeds, so every subject is tested in the same fold
as in `GAT_ds004504.ipynb`, by a forest that saw none of that subject's windows.
A subject's score is the mean probability over their windows. The headline
number is **pooled over all 65 subjects**, at window and at subject level.

**Rows it is compared with** (all on the same folds; Cell 9 reads the graph
models automatically when their protocol matches):

| model | subject acc | subject AUC |
|---|---|---|
| band-power logistic regression (Cell 5) | 0.7538 | 0.8812 |
| MFE-FCGAT (`results_gat/`, lr 3e-3) | 0.8615 | 0.9272 |
| MFE-FCGCN (`results_gcn/`) | shown in Cell 9 once run | |

**Why the floor here is not 0.8769 / 0.8908.** The Cell 4 floor in
`GAT_ds004504.ipynb` and `GCN_ds004504.ipynb` is scored on different folds
(`random_state=SEED`) from the ones their networks are tested on
(`SEED + 1000`). Here the logistic regression uses the same folds as every
other row. The identical model scores 0.8769 / 0.8908 on the seed-42 folds and
0.7538 / 0.8812 on these: **accuracy moved 0.12 with the fold draw, AUC 0.01.**

The notebook needs `subject_cache/` next to it (65 `.npz` files, built by
`mfe-fcgcn.ipynb`). It runs on CPU in well under a minute; no GPU or PyTorch
needed. **Cell 2 is normally the only cell you need to edit.**

In [ ]:
# ===========================================================================
# CELL 1 — IMPORTS AND PATHS
# ===========================================================================
import json, time, warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
from matplotlib.colors import LinearSegmentedColormap
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, recall_score, roc_auc_score, roc_curve
from sklearn.model_selection import StratifiedGroupKFold, StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")

WORK_DIR   = Path.cwd()
CACHE_DIR  = WORK_DIR / "subject_cache"      # point elsewhere if the cache lives elsewhere
RESULT_DIR = WORK_DIR / "results_rf"
RESULT_DIR.mkdir(parents=True, exist_ok=True)

print(f"scikit-learn {sklearn.__version__}")
print(f"cache   : {CACHE_DIR}  ({len(list(CACHE_DIR.glob('sub-*.npz')))} files)")
print(f"results : {RESULT_DIR}")

In [ ]:
# ===========================================================================
# CELL 2 — CONFIGURATION  (the only cell you normally edit)
# ===========================================================================
SEED = 42

# --- data: fixed by the cache ----------------------------------------------
BAND_NAMES = ["Delta", "Theta", "Alpha", "Beta", "Gamma"]
CHANNELS   = ["Fp1", "Fp2", "F3", "F4", "C3", "C4", "P3", "P4", "O1", "O2",
              "F7", "F8", "T3", "T4", "T5", "T6", "Fz", "Cz", "Pz"]   # cache order (mfe-fcgcn.ipynb)
N_NODES    = 19            # electrodes
N_SUBJECTS = 65            # sub-001..036 AD (label 1), sub-037..065 HC (label 0)
N_AD       = 36

# --- model -------------------------------------------------------------------
# scikit-learn defaults apart from the tree count. Nothing here was tuned on
# these folds, so the baseline cannot have been fitted to its own test subjects.
N_TREES          = 500     # more trees only smooth the probabilities; 500 is plenty for 95 features
MAX_FEATURES     = "sqrt"  # ~10 of the 95 features are tried at each split (default)
MIN_SAMPLES_LEAF = 1       # default: fully grown trees
N_JOBS           = -1      # use every CPU core

# --- protocol (keep identical across GCN_ds004504, GAT_ds004504 and this) ---
# "subject": no window from a test subject is ever used in training (honest).
# "window" : windows are shuffled freely, so overlapping windows of the same
#            person land on both sides of the split (the paper's ~95 % setting).
SPLIT_LEVEL = "subject"
N_FOLDS     = 5
N_REPEATS   = 1            # use 3+ before any number goes into a write-up

assert SPLIT_LEVEL in ("subject", "window")
assert len(CHANNELS) == N_NODES
np.random.seed(SEED)
print(f"Random forest | {N_TREES} trees, max_features {MAX_FEATURES}, min_samples_leaf {MIN_SAMPLES_LEAF} | "
      f"{N_REPEATS} x {N_FOLDS}-fold, {SPLIT_LEVEL}-level")

In [ ]:
# ===========================================================================
# CELL 3 — LOAD THE FEATURE CACHE
# ===========================================================================
# One .npz per subject; one row per 20 s window (75 % overlap). Only two arrays
# are needed here, and .npz members load lazily, so the PSDs and graphs that
# the graph notebooks use are never read:
#   bp   (n, 5, 19)   mean log10 band power, per band per electrode
#   y    (n,)         1 = AD, 0 = HC
bp_parts, y_parts, sid_parts = [], [], []

for i in range(1, N_SUBJECTS + 1):
    sid = f"sub-{i:03d}"
    with np.load(CACHE_DIR / f"{sid}.npz") as z:
        assert z["bp"].shape[1:] == (len(BAND_NAMES), N_NODES), sid
        bp_parts.append(z["bp"]); y_parts.append(z["y"])
        sid_parts.append(np.full(len(z["y"]), sid))
    # a re-cached subject must never silently flip class
    assert int(y_parts[-1][0]) == (1 if i <= N_AD else 0), f"{sid}: wrong label"

BP  = np.concatenate(bp_parts).astype(np.float32)
Y   = np.concatenate(y_parts).astype(np.int64)
SID = np.concatenate(sid_parts)
del bp_parts, y_parts, sid_parts

print(f"windows  : {len(Y):,}  (AD {int((Y == 1).sum()):,} / HC {int((Y == 0).sum()):,})")
print(f"subjects : {len(np.unique(SID))}  (AD {N_AD} / HC {N_SUBJECTS - N_AD})")
print(f"  bp       {BP.shape}")

In [ ]:
# ===========================================================================
# CELL 4 — SPLITS AND METRICS  (copied unchanged from GAT_ds004504)
# ===========================================================================
def make_splits(level=SPLIT_LEVEL, n_folds=N_FOLDS, n_repeats=N_REPEATS, seed=SEED):
    """The outer K-fold picks the test set; 1/5 of the remainder becomes validation.

    Within each repeat, every window (and so every subject) is tested exactly once.
    """
    splits, idx = [], np.arange(len(Y))
    for r in range(1, n_repeats + 1):
        rs = seed + 1000 * r
        outer = (StratifiedGroupKFold(n_folds, shuffle=True, random_state=rs).split(idx, Y, groups=SID)
                 if level == "subject" else
                 StratifiedKFold(n_folds, shuffle=True, random_state=rs).split(idx, Y))
        for f, (trainval, test) in enumerate(outer, start=1):
            inner = StratifiedKFold(5, shuffle=True, random_state=rs + f)
            if level == "subject":
                # Hold out validation SUBJECTS, so early stopping is never
                # judged on people the model is training on.
                subs = np.unique(SID[trainval])
                labs = np.array([Y[SID == s][0] for s in subs])
                _, va = next(inner.split(subs, labs))
                is_val = np.isin(SID[trainval], subs[va])
                train, val = trainval[~is_val], trainval[is_val]
                tr_s, va_s, te_s = set(SID[train]), set(SID[val]), set(SID[test])
                assert not (tr_s & te_s or va_s & te_s or tr_s & va_s), "SUBJECT LEAKAGE"
            else:
                tr, va = next(inner.split(trainval, Y[trainval]))
                train, val = trainval[tr], trainval[va]
            splits.append(dict(repeat=r, fold=f, train=train, val=val, test=test))
    return splits


def _metrics(y, p):
    pred = (p >= 0.5).astype(int)
    return dict(acc=accuracy_score(y, pred),
                sens=recall_score(y, pred, zero_division=0),
                spec=recall_score(1 - y, 1 - pred, zero_division=0),
                f1=f1_score(y, pred, zero_division=0),
                auc=roc_auc_score(y, p) if len(np.unique(y)) == 2 else np.nan)


def score(idx, prob):
    """Window-level (w_*) and subject-level (s_*) metrics.
    A subject's probability is the mean over its windows."""
    sid, y = SID[idx], Y[idx]
    subs = np.unique(sid)
    s_p = np.array([prob[sid == s].mean() for s in subs])
    s_y = np.array([y[sid == s][0] for s in subs])
    return {**{f"w_{k}": v for k, v in _metrics(y, prob).items()},
            **{f"s_{k}": v for k, v in _metrics(s_y, s_p).items()}}


SPLITS = make_splits()
_ns = lambda i: len(np.unique(SID[i]))
for s in SPLITS:
    print(f"r{s['repeat']} f{s['fold']}: train {len(s['train']):5d}w/{_ns(s['train']):2d}s | "
          f"val {len(s['val']):4d}w/{_ns(s['val']):2d}s | test {len(s['test']):4d}w/{_ns(s['test']):2d}s")
_tested = np.concatenate([s["test"] for s in SPLITS if s["repeat"] == 1])
assert len(_tested) == len(np.unique(_tested)) == len(Y)
print(f"each repeat tests all {len(Y):,} windows / {_ns(_tested)} subjects exactly once")

In [ ]:
# ===========================================================================
# CELL 5 — THE BAR: band power -> logistic regression, on the same folds
# ===========================================================================
# The same 95 numbers the forest uses, with a linear model instead. It is
# fitted on the same train + validation subjects and tested on the same test
# subjects as the forest, so the two rows differ only in the classifier.
feat = BP.reshape(len(BP), -1)
FLOOR_OOF = np.full((N_REPEATS, len(Y)), np.nan)
for sp in SPLITS:
    fit = np.concatenate([sp["train"], sp["val"]])
    clf = make_pipeline(StandardScaler(), LogisticRegression(C=0.1, max_iter=2000))
    FLOOR_OOF[sp["repeat"] - 1, sp["test"]] = clf.fit(feat[fit], Y[fit]).predict_proba(feat[sp["test"]])[:, 1]

FLOOR = pd.DataFrame([score(np.arange(len(Y)), FLOOR_OOF[r]) for r in range(N_REPEATS)]).mean()
print(f"linear floor, same folds : window acc {FLOOR['w_acc']:.4f} auc {FLOOR['w_auc']:.4f}  |  "
      f"subject acc {FLOOR['s_acc']:.4f} auc {FLOOR['s_auc']:.4f}")

---
## Model

Things to know before reading the code:

- **Same 95 numbers as the linear floor.** The forest and the Cell 5 logistic
  regression see identical features, are fitted on identical subjects and are
  scored identically. The only difference between them is the classifier.
  Against the GAT, the difference is the input: no spectral shape inside each
  band and no connectivity graphs.
- **Fitted on train + validation subjects.** The graph models need validation
  subjects to choose an epoch. A forest has no epoch to choose, so those
  subjects join its training set. The forest learns from about 52 subjects
  where the GAT learns from about 41. That makes the bar harder to clear, which
  is the right direction for a baseline.
- **No out-of-bag score.** Windows overlap by 75 %, so a window left out of a
  tree's bootstrap sample has near-copies inside it. Its out-of-bag estimate
  would behave like a window-level split. Only the subject-grouped test folds
  are used.
- **Trees cannot take ratios.** Each split tests one feature against a
  threshold. Classic EEG slowing markers such as theta/alpha are differences of
  two log features here. Logistic regression gets them from one weight vector;
  a tree has to approximate them with a staircase of splits. In exchange, a
  forest can pick up interactions and thresholds that a linear model cannot.

In [ ]:
# ===========================================================================
# CELL 6 — MODEL
# ===========================================================================
FEAT       = BP.reshape(len(BP), -1)                                       # (n, 95)
FEAT_NAMES = np.array([f"{b}_{c}" for b in BAND_NAMES for c in CHANNELS])  # same order as the reshape
assert FEAT[0, N_NODES + 2] == BP[0, 1, 2] and FEAT_NAMES[N_NODES + 2] == "Theta_F3"


def make_model(seed):
    return RandomForestClassifier(n_estimators=N_TREES, max_features=MAX_FEATURES,
                                  min_samples_leaf=MIN_SAMPLES_LEAF, n_jobs=N_JOBS,
                                  random_state=seed)


_i = np.r_[:100, len(Y) - 100:len(Y)]                  # 100 AD + 100 HC windows
_m = make_model(SEED).fit(FEAT[_i], Y[_i])
print(f"features   : {FEAT.shape[1]}  ({len(BAND_NAMES)} bands x {N_NODES} electrodes)")
print(f"smoke test : P(AD) {_m.predict_proba(FEAT[_i]).shape}, classes {_m.classes_.tolist()}")
del _i, _m

In [ ]:
# ===========================================================================
# CELL 7 — TRAINING
# ===========================================================================
def train_fold(split):
    """Fit on the train + validation subjects (no epoch to select), predict test."""
    seed = SEED + 1000 * split["repeat"] + split["fold"]
    fit  = np.concatenate([split["train"], split["val"]])
    model = make_model(seed).fit(FEAT[fit], Y[fit])
    assert model.classes_.tolist() == [0, 1]           # column 1 of predict_proba is AD
    return dict(fit=fit, test_prob=model.predict_proba(FEAT[split["test"]])[:, 1],
                importance=model.feature_importances_)


print("training function ready")

In [ ]:
# ===========================================================================
# CELL 8 — CROSS-VALIDATION
# ===========================================================================
rows, importances = [], []
OOF = np.full((N_REPEATS, len(Y)), np.nan)        # out-of-fold P(AD), one row per repeat
t0 = time.time()

for sp in SPLITS:
    ts = time.time()
    r = train_fold(sp)
    OOF[sp["repeat"] - 1, sp["test"]] = r["test_prob"]
    importances.append(r["importance"])
    m = score(sp["test"], r["test_prob"])
    rows.append(dict(repeat=sp["repeat"], fold=sp["fold"], fit_windows=len(r["fit"]),
                     fit_subjects=_ns(r["fit"]), seconds=round(time.time() - ts, 1), **m))
    print(f"r{sp['repeat']} f{sp['fold']}  fit on {len(r['fit']):4d}w/{_ns(r['fit'])}s  "
          f"window acc {m['w_acc']:.4f} auc {m['w_auc']:.4f}  |  "
          f"subject acc {m['s_acc']:.4f} auc {m['s_auc']:.4f}  [{rows[-1]['seconds']}s]", flush=True)

FOLDS = pd.DataFrame(rows)
FOLDS.to_csv(RESULT_DIR / "fold_results.csv", index=False)
np.save(RESULT_DIR / "oof_prob.npy", OOF)

# Mean decrease in impurity, averaged over every fitted forest.
IMPORTANCE = pd.Series(np.mean(importances, axis=0), index=FEAT_NAMES, name="importance")
IMPORTANCE.to_csv(RESULT_DIR / "feature_importance.csv")

# Pooled out-of-fold score: each repeat scores all 65 subjects once, each by a
# model that never saw them. This is the number to quote.
POOLED = pd.DataFrame([score(np.arange(len(Y)), OOF[r]) for r in range(N_REPEATS)])
P    = POOLED.mean()
P_SD = POOLED.std(ddof=1) if N_REPEATS > 1 else P * 0
pm = lambda k: f"{P[k]:.4f}" + (f" +- {P_SD[k]:.4f}" if N_REPEATS > 1 else "")

print(f"\nPOOLED  window  acc {pm('w_acc')}  auc {pm('w_auc')}")
print(f"POOLED  subject acc {pm('s_acc')}  auc {pm('s_auc')}  "
      f"sens {P['s_sens']:.4f}  spec {P['s_spec']:.4f}   [{(time.time() - t0) / 60:.1f} min]")

_ = (RESULT_DIR / "summary.json").write_text(json.dumps(dict(
    model="RandomForest", pooled=P.to_dict(), pooled_sd=P_SD.to_dict(), floor=FLOOR.to_dict(),
    config=dict(FEATURES="bp: mean log10 band power, 5 bands x 19 electrodes",
                N_TREES=N_TREES, MAX_FEATURES=MAX_FEATURES, MIN_SAMPLES_LEAF=MIN_SAMPLES_LEAF,
                SPLIT_LEVEL=SPLIT_LEVEL, N_FOLDS=N_FOLDS, N_REPEATS=N_REPEATS, SEED=SEED)), indent=2))

In [ ]:
# ===========================================================================
# CELL 9 — RESULTS
# ===========================================================================
# The graph models are shown too if their notebooks have been run next to this
# one with the same protocol (same splits, so the comparison is like-for-like).
PROTOCOL = dict(SEED=SEED, N_FOLDS=N_FOLDS, N_REPEATS=N_REPEATS, SPLIT_LEVEL=SPLIT_LEVEL)
OTHERS = {}
for name, folder in [("MFE-FCGAT", "results_gat"), ("MFE-FCGCN", "results_gcn")]:
    path = WORK_DIR / folder / "summary.json"
    if not path.exists():
        print(f"note: {folder}/ not found, run its notebook to add {name}")
        continue
    s = json.loads(path.read_text())
    if any(s["config"].get(k) != v for k, v in PROTOCOL.items()):
        print(f"note: {folder}/ used a different protocol, so it is left out")
        continue
    OTHERS[name] = dict(dir=WORK_DIR / folder, lr=s["config"]["LR"], pooled=pd.Series(s["pooled"]))

table  = [("Random forest (this run)", P)]
table += [(f"{name} (lr {o['lr']:g})", o["pooled"]) for name, o in OTHERS.items()]
table += [("logistic regression (Cell 5)", FLOOR)]

print("\n" + "=" * 86)
print(f"{'pooled over 65 subjects':32s}{'--- window ---':>18s}{'---------- subject ----------':>36s}")
print(f"{'':32s}{'acc':>9s}{'AUC':>9s}{'acc':>9s}{'AUC':>9s}{'sens':>9s}{'spec':>9s}")
print("-" * 86)
for name, p in table:
    print(f"{name:32s}{p['w_acc']:9.4f}{p['w_auc']:9.4f}"
          f"{p['s_acc']:9.4f}{p['s_auc']:9.4f}{p['s_sens']:9.4f}{p['s_spec']:9.4f}")
print("=" * 86)
print(f"forest vs linear floor: acc {P['s_acc'] - FLOOR['s_acc']:+.4f}   AUC {P['s_auc'] - FLOOR['s_auc']:+.4f}"
      "   (under ~0.03 AUC is noise at N_REPEATS = 1)")

# --- which features does the forest use? --------------------------------------
# Impurity importance shares credit among correlated features, and neighbouring
# electrodes carry nearly the same band power, so read the band totals first.
by_band = IMPORTANCE.groupby(lambda k: k.split("_")[0]).sum().reindex(BAND_NAMES)
print("\nimportance by band : " + "   ".join(f"{b} {v:.3f}" for b, v in by_band.items()))
print("top 10 features    : " + ", ".join(f"{k} {v:.3f}" for k, v in IMPORTANCE.nlargest(10).items()))

# --- which subjects are wrong? ------------------------------------------------
_p = np.nanmean(OOF, axis=0)
PER_SUBJECT = pd.DataFrame([dict(subject=s, label="AD" if Y[SID == s][0] else "HC",
                                 p_ad=_p[SID == s].mean()) for s in np.unique(SID)])
PER_SUBJECT["called"]  = np.where(PER_SUBJECT.p_ad > 0.5, "AD", "HC")
PER_SUBJECT["correct"] = PER_SUBJECT.called == PER_SUBJECT.label
PER_SUBJECT.to_csv(RESULT_DIR / "per_subject.csv", index=False)
print(f"\n{PER_SUBJECT.correct.sum()}/{len(PER_SUBJECT)} subjects correct; misclassified:")
print(PER_SUBJECT[~PER_SUBJECT.correct].sort_values("p_ad").to_string(
    index=False, float_format=lambda v: f"{v:.3f}"))

# --- plots --------------------------------------------------------------------
# A model keeps its colour whether or not the others are present.
COLOR   = {"Random forest": "#2a78d6", "MFE-FCGAT": "#eb6834", "MFE-FCGCN": "#1baf7a"}
INK, MUTED, GRID, SURFACE = "#0b0b0b", "#898781", "#e1e0d9", "#fcfcfb"
BLUES = LinearSegmentedColormap.from_list("blues", ["#cde2fb", "#9ec5f4", "#6da7ec", "#3987e5",
                                                    "#256abf", "#184f95", "#0d366b"])
# electrodes front to back, left to right, so the map reads like the scalp
SCALP = ["Fp1", "Fp2", "F7", "F3", "Fz", "F4", "F8", "T3", "C3", "Cz",
         "C4", "T4", "T5", "P3", "Pz", "P4", "T6", "O1", "O2"]
assert sorted(SCALP) == sorted(CHANNELS)
subject_prob = lambda oof: np.array([np.nanmean(oof, axis=0)[SID == s].mean() for s in PER_SUBJECT.subject])

with plt.rc_context({"axes.edgecolor": "#c3c2b7", "axes.labelcolor": INK, "text.color": INK,
                     "xtick.color": MUTED, "ytick.color": MUTED, "figure.facecolor": SURFACE,
                     "axes.facecolor": SURFACE, "grid.color": GRID, "grid.linewidth": 0.6}):
    fig, ax = plt.subplots(1, 2, figsize=(15, 4.8), gridspec_kw=dict(width_ratios=[1, 1.9]))

    _y = (PER_SUBJECT.label == "AD").astype(int)
    ax[0].plot(*roc_curve(_y, PER_SUBJECT.p_ad)[:2], lw=2, color=COLOR["Random forest"],
               label=f"Random forest  AUC {roc_auc_score(_y, PER_SUBJECT.p_ad):.3f}")
    for name, o in OTHERS.items():
        _o = np.load(o["dir"] / "oof_prob.npy")
        assert _o.shape[1] == len(Y), f"{o['dir'].name}: built from a different cache"
        _op = subject_prob(_o)
        ax[0].plot(*roc_curve(_y, _op)[:2], lw=1.5, color=COLOR[name],
                   label=f"{name}  AUC {roc_auc_score(_y, _op):.3f}")
    _fp = subject_prob(FLOOR_OOF)
    ax[0].plot(*roc_curve(_y, _fp)[:2], lw=1.2, ls="--", color="#52514e",
               label=f"linear floor  AUC {roc_auc_score(_y, _fp):.3f}")
    ax[0].plot([0, 1], [0, 1], color=MUTED, lw=0.8, ls=":")
    ax[0].set(xlabel="1 - specificity", ylabel="sensitivity", title="Subject-level ROC (65 subjects)")
    ax[0].legend(frameon=False, loc="lower right"); ax[0].grid(True)

    heat = IMPORTANCE.values.reshape(len(BAND_NAMES), N_NODES)[:, [CHANNELS.index(c) for c in SCALP]]
    mesh = ax[1].pcolormesh(heat, cmap=BLUES, vmin=0, edgecolors=SURFACE, linewidth=1.5)
    ax[1].set_xticks(np.arange(N_NODES) + 0.5, labels=SCALP)
    ax[1].set_yticks(np.arange(len(BAND_NAMES)) + 0.5, labels=BAND_NAMES)
    ax[1].invert_yaxis()
    ax[1].tick_params(length=0)
    for side in ax[1].spines.values():
        side.set_visible(False)
    ax[1].set(xlabel="electrode (front to back)",
              title="Random forest: feature importance (mean decrease in impurity, all folds)")
    fig.colorbar(mesh, ax=ax[1], fraction=0.025, pad=0.01).outline.set_visible(False)

    plt.tight_layout()
    plt.savefig(RESULT_DIR / "summary.png", dpi=140)
    plt.show()

---
## Reading the result

- **Quote the pooled row only.** Each fold has 13 test subjects, so a single
  subject is worth 7.7 accuracy points. Single-fold numbers in this project
  came out up to 0.09 too optimistic.
- **Trust AUC over accuracy.** Accuracy uses a fixed 0.5 cut, and calibration
  shifts between groups of subjects. The logistic regression's subject
  accuracy moves by 0.12 between two fold draws while its AUC moves by 0.01.
  A forest's probabilities are vote shares, which are often squeezed toward the
  middle, so its accuracy at 0.5 is even less stable.
- **At `N_REPEATS = 1`, AUC differences under ~0.03 are noise.** A forest is
  cheap, but the table only shows models run with the same `N_REPEATS`, so
  change it in every notebook together.
- **Compare rows only on the same folds.** Every row in the Cell 9 table is.
  The 0.8769 / 0.8908 floor quoted in the graph notebooks is not: it was scored
  on a different fold draw.
- **Forest vs floor** isolates the classifier: same features, same subjects,
  same scoring. **Forest vs GAT** isolates the input: the GAT also sees the
  spectral shape inside each band and the connectivity graphs.
- **Window-level numbers** are the ones to set against papers that score
  per epoch. Subject-level numbers are for papers that score per person.
- **Feature importance** is computed on training windows and shared among
  correlated features. Read the band totals and the broad pattern on the map,
  not the rank of one electrode.